# Suite ANY — Values of any type

`Schemas.OfAny.Value` is the kind of a value of any type: an `OfAny.Builder` with several types selected restricts it
to them, and one with none selected leaves it unrestricted. A restricted value is written bare, its type the one whose
plain form it takes; an unrestricted one is tagged with its type, a scalar's token name or a registered schema's name,
and is assigned as a value, built standalone through the store where it is not a scalar.

In [ ]:
import { Comparison as C, JSON, Modules, Plain, Proxies, Schemas as S, Validators, YAML } from "@mbse/schemas/Framework";
import { DecodeError, ReadOnlyError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, map, raises, root_of, same, text } from "./support.js";

const Int = new S.OfScalar.Data(BigInt);
const Text = new S.OfScalar.Data(String);
const Float = new S.OfScalar.Data(Number);
const Bytes = new S.OfScalar.Data(Uint8Array);
const Phone = new S.OfObject.Builder().name("Phone").properties(text("number")).create();
const Code = new S.OfUnion.Builder().name("Code").branches((b) => b.name("n").of(Int), (b) => b.name("s").of(Text)).create();
const Stamp = new S.OfIntersection.Builder().name("Stamp").parts((p) => p.name("at").of(Int), (p) => p.name("by").of(Text)).create();
const Tags = new S.OfContainer.Builder().name("Tags").index(Int).element(Text).create();
const Labels = new S.OfContainer.Builder().name("Labels").index(Text).element(Text).create(); // a dict by text: a mapping
const Cell = new S.OfObject.Builder().properties(text("r", BigInt), text("c", BigInt)).create();
const Grid = new S.OfContainer.Builder().index(Cell).element(Int).create(); // a tensor: nested arrays, or flat with its shape
const One = S.OfAny.resolve((t) => t.as_scalar(BigInt).as_scalar(String).as_object(Phone)) as S.OfAny.Value;
const Free = S.OfAny.resolve((t) => t) as S.OfAny.Value;
const Card = new S.OfObject.Builder().name("Card").ref(String.raw`card-(\d+)`).properties(
  (p) => p.name("one").of(One), (p) => p.name("free").of(Free),
  (p) => p.name("many").of((t) => t.as_container((c) => c.index(Int).element(Free))),
  (p) => p.name("tags").of((t) => t.as_scalar(Boolean).as_container(Tags))).create();
const store = new Proxies.OfStore();
for (const schema of [Phone, Code, Stamp, Tags, Labels, Card]) store.register(schema);
const B: any = store;

/** What a property holds, read through its visitor: a scalar, or a visitor's schema's name; null where absent. */
function held(prop: any): unknown {
  const found: unknown[] = [];
  const named = (v: any) => {
    if (v.as_schema() !== null) found.push(v.as_schema().name);
    return v;
  };
  prop.as_value((a: any) => a.as_scalar((s: any) => {
    if (s.get() !== null) found.push(s.get());
    return s;
  }).as_object(named).as_union(named).as_intersection(named).as_container(named));
  return found.length > 0 ? found[0] : null;
}

/** A schema's plain forms, sorted. */
const forms = (schema: unknown) => [...S.plain_forms(schema)].sort();

## ANY-01 · An `OfAny` restricted to the types selected, or to none; its types told apart on the wire

In [ ]:
{
  assert(One instanceof S.OfAny.Value && One.types.length === 3 && One.types[0]!.equals(Int) && One.types[1]!.equals(Text)
    && One.types[2] === Phone && One.validate().length === 0);
  assert(Free.types.length === 0 && Free.validate().length === 0 && Card.validate().length === 0);
  assert(S.OfAny.resolve((t) => t.as_scalar(BigInt)).equals(Int)); // one type selected: that type, as ever
  assert(equal([forms(Float), forms(Bytes), forms(Labels), forms(Tags), forms(Grid)],
    [["float", "text"], ["text"], ["mapping"], ["list"], ["list", "mapping"]])); // NaN is text, bytes base64, a tensor flat
  assert(same(forms(One), ["int", "mapping", "text"]) && same(forms(Free), ["mapping"])); // unrestricted: tagged
  assert(same(forms(new S.OfScalar.Data(new S.OfScalar.Token("ccpp", "x"))), ["bool", "float", "int", "list", "mapping", "text"]));
  const loop = new S.OfApply.Data();
  loop.of = loop;
  assert(forms(loop).length === 0); // an application that applies itself has no values
  assert(same(new S.OfAny.Value([loop, new S.OfScalar.Data("weird")]).validate(), [
    "a schema: an application cannot apply itself", "a schema: unsupported scalar type 'weird'"]));
  assert(same(new S.OfAny.Value([Tags, Grid]).validate(), ["an OfAny's types Tags and a container are not told apart on the wire"]));
  assert(same(new S.OfAny.Value([Cell, Labels]).validate(), [
    "an OfAny's types a value object and Labels are not told apart on the wire"]));
  assert(One.equals(One) && !One.equals(Free)); // schemas by identity
  const Either = new S.OfUnion.Builder().branches((b) => b.name("a").of(Int), (b) => b.name("b").of(Text)).create();
  const Both = new S.OfIntersection.Builder().parts((p) => p.name("a").of(Int), (p) => p.name("b").of(Text)).create();
  assert(same(new S.OfAny.Value([Float, Text, Phone, Code, Labels, Either, Both, Tags]).validate(), [
    "an OfAny's types float and str are not told apart on the wire",
    "an OfAny's types Phone and Code are not told apart on the wire",
    "an OfAny's types Phone and Labels are not told apart on the wire",
    "an OfAny's types Phone and a union are not told apart on the wire",
    "an OfAny's types Phone and an intersection are not told apart on the wire",
    "an OfAny's types Code and Labels are not told apart on the wire",
    "an OfAny's types Code and a union are not told apart on the wire",
    "an OfAny's types Code and an intersection are not told apart on the wire",
    "an OfAny's types Labels and a union are not told apart on the wire",
    "an OfAny's types Labels and an intersection are not told apart on the wire",
    "an OfAny's types a union and an intersection are not told apart on the wire"]));
  assert(same(new S.OfAny.Value([3n, One, new S.OfScalar.Data(BigInt, { name: "1x" }), Grid]).validate(), [
    "an OfAny's types are schemas, got 3", "an OfAny's types are schemas of other kinds, not an OfAny",
    "1x: name '1x' is not identifiers separated by dots, e.g. 'crm.Contact'"]));
  const Thing = new S.OfObject.Builder().name("Thing").ref().create();
  assert(same(new S.OfObject.Builder().properties((p) => p.name("x").of((t) => t.as_scalar(BigInt).as_object(Thing))).create()
    .validate(), ["property 'x': a reference object schema cannot be a property's type"])); // a value only
}

## ANY-02 · Proxies: a restricted value written through the visitors as its type is, inferred; an unrestricted one given as a value, built standalone where it is not a scalar

In [ ]:
let card = B.Card().one(3n).free("x").create();
assert(held(card.one) === 3n && held(card.free) === "x" && card.one.get() === 3n);
card = B.Card().one((p: any) => p.as_object((o: any) => o.number("1"))).tags( // the one type of its kind
  (p: any) => p.as_value((a: any) => a.as_container((t: any) => t.at(0n).as_scalar("a")))).create();
assert(held(card.one) === "Phone" && held(card.tags) === "Tags" && card.one.get() === null);
raises(TypeError, () => B.Card().one(B.Tags().create()), "property 'one' does not hold a value of Tags");
raises(TypeError, () => Proxies.OfObject.Builder(store, Phone, "Phone"),
  "'Phone' is a value object schema; build a value through its owner, or standalone through the store");
raises(ReadOnlyError, () => B.Card().one((p: any) => p.as_value((a: any) => a.as_union((u: any) => u.n(1n)))),
  "property 'one' does not hold a union value");
// Unrestricted: a scalar, or a value, standalone or another's; a Spec has no schema to build by.
const phone = B.Phone().number("2").create();
let tags = B.Tags();
tags.at(0n).as_scalar("t");
tags = tags.create();
assert(Proxies.owner_of(phone) === null && tags.as_schema() === Tags);
const values = [phone, B.Code().n(1n).create(), B.Stamp().at(1n).by("me").create(), tags, card.one];
assert(same(values.map((value) => held(B.Card().free(value).create().free)), ["Phone", "Code", "Stamp", "Tags", "Phone"]));
assert(held(B.Card().free((p: any) => p.as_value((a: any) => a.as_object(card.one))).create().free) === "Phone");
{
  const fresh = B.Phone().number("3").create();
  const [first, second] = [B.Card().free(fresh).create(), B.Card().free(fresh).create()];
  const phones: any[] = [];
  for (const each of [first, second]) each.free.as_value((a: any) => a.as_object((o: any) => phones.push(o) && o));
  assert(phones[0] === fresh && phones[1] !== fresh && Proxies.owner_of(fresh) === first); // adopted once, then copied
  const unbuilt = "property 'free' holds a value of any type: give a value, which carries its schema, not a Spec";
  raises(ReadOnlyError, () => B.Card().free((p: any) => p.as_value((a: any) => a.as_object((o: any) => o.number("1")))), unbuilt);
  raises(ReadOnlyError, () => B.Card().free((p: any) => p.as_value((a: any) => a.as_container((c: any) => c.remove(0n)))), unbuilt);
  const read: unknown[] = []; // a Spec of another kind than the value held reads it as empty, and writes nothing
  B.Card().one(3n).one((p: any) => p.as_value((a: any) => a.as_object((o: any) => read.push(o.as_schema()) && o))).create();
  assert(same(read, [null]));
  const found: unknown[] = [];
  card.free.as_value((a: any) => a.as_scalar((s: any) => found.push([s.as_schema(), s.get()]) && s));
  assert(equal(found, [[Free, null]])); // nothing held: a scalar of any type, absent
  const listed = B.Card().many((p: any) => p.as_value((a: any) => a.as_container(
    (c: any) => c.at(0n).as_scalar(1n) && c.at(1n).as_object(phone)))).create();
  const elements: [unknown, unknown][] = [];
  listed.many.as_value((a: any) => a.as_container((c: any) => c.as_entries((it: Iterable<[unknown, unknown]>) => elements.push(...it))));
  assert(same(elements.map(([index]) => index), [0n, 1n]));
  raises(TypeError, () => B.Card().one(listed.many), "property 'one' does not hold a value of a container");
}

## ANY-03 · Snapshots: a restricted value bare, an unrestricted one tagged, read back; malformed ones refused, with a path

In [ ]:
const other = new Proxies.OfStore();
for (const schema of [Phone, Code, Stamp, Tags, Labels, Card]) other.register(schema);
const many = (c: any) => {
  c.at(0n).as_scalar(2.5);
  c.at(1n).as_object(phone);
  c.at(3n).as_container(tags); // nothing at 2
  return c;
};
card = B.Card().one((p: any) => p.as_object((o: any) => o.number("1"))).free(B.Code().s("c").create())
  .many((p: any) => p.as_value((a: any) => a.as_container(many))).tags(true).create();
assert(equal(root_of(Plain.ToPlain(store).OfObject(Card, card)), map({ one: { number: "1" }, free: { Code: { s: "c" } },
  tags: true, many: [{ float: 2.5 }, { Phone: { number: "2" } }, null, { Tags: ["t"] }] })));
for (const [encode, decode] of [[JSON.ToJSON, JSON.FromJSON], [YAML.ToYAML, YAML.FromYAML]] as const) {
  const written = (encode as any)(store).OfObject(Card, card);
  assert((encode as any)(other).OfObject(Card, (decode as any)(other).OfObject(Card, written)) === written);
}
assert(equal(root_of(Plain.ToPlain(store).OfObject(Card, B.Card().one("s").free(new Uint8Array([1])).create())),
  map({ one: "s", free: { bytes: "AQ==" } })));
assert(Plain.FromPlain(store).OfObject(Card, Plain.ToPlain(store).OfObject(Card, card)) === card); // reloading: an update
{
  const back: any = Plain.FromPlain(other).OfObject(Card, map({ root: "Card/card-8", objects: { "Card/card-8": { one: 3n, tags: ["a"] } } }));
  assert(held(back.one) === 3n && held(back.tags) === "Tags"); // each read as the one type its plain form fits
  const text_: any = Plain.FromPlain(other).OfObject(Card, map({ root: "Card/card-7", objects: { "Card/card-7": { one: "s" } } }));
  assert(held(text_.one) === "s");
}
const refused = (values: Record<string, unknown>, message: string) => raises(DecodeError,
  () => Plain.FromPlain(other).OfObject(Card, map({ root: "Card/card-9", objects: { "Card/card-9": values } })),
  `$.objects["Card/card-9"]${message}`);
refused({ one: true }, ".one: a value of any type takes the plain form of one of its types, got bool");
refused({ one: 2.5 }, ".one: a value of any type takes the plain form of one of its types, got float");
refused({ one: null }, ".one: a value of any type takes the plain form of one of its types, got NoneType");
refused({ free: 3n }, ".free: a value of any type is {type: value}, its type a scalar's token name or a registered schema's name");
refused({ free: { Nope: 1n } }, ".free.Nope: no schema registered as 'Nope'");
refused({ free: { Card: {} } }, ".free.Card: 'Card' is not a value's schema");
refused({ free: { int: "x" } }, ".free.int: expected int, got str");
raises(TypeError, () => Plain.ToPlain(store).OfObject(Card, B.Card().one(2.5).create()),
  "a value of any type holds one of its types; float is not one"); // builders do not check scalars' types
const Point = new S.OfObject.Builder().name("Point").properties(text("x", BigInt)).create(); // not registered
const Spot = new S.OfObject.Builder().name("Spot").ref().properties((p) => p.name("at").of(Point)).create();
store.register(Spot);
const loose = B.Card().free(B.Spot().at((p: any) => p.as_object((o: any) => o.x(1n))).create().at).create();
raises(TypeError, () => Plain.ToPlain(store).OfObject(Card, loose), "a value of any type is of a registered schema; 'Point' is not registered");

## ANY-04 · Validation and comparison under the type a value holds; uniqueness by value

In [ ]:
{
  const validate = Validators.Validate(store);
  assert(validate(Card, card).length === 0);
  assert(same(validate(Card, B.Card().one(2.5).many((p: any) => p.as_value((a: any) => a.as_container(
    (c: any) => c.at(0n).as_object(phone)))).create()), ["Card#0.one: float is not one of its types"]));
  assert(same(validate(Card, loose), ["Card#0.free: a value of any type is of a registered schema; 'Point' is not registered"]));
  assert(same(validate(Card, B.Card().free(B.Code().n("x").create()).create()), ["Card#0.free.n: expected int, got str"]));
  const compare = (a: any, b: any) => new C.OfObject(Card, a).compare(new C.OfObject(Card, b));
  assert(compare(B.Card().one(3n).free(phone).create(), B.Card().one(3n).free(B.Phone().number("2").create()).create()) === 0);
  assert(compare(B.Card().one(3n).create(), B.Card().one("3").create()) === null); // of different types: incomparable
  assert(compare(B.Card().free(phone).create(), B.Card().free(B.Code().s("2").create()).create()) === null);
  const Holds = new S.OfRelation.Builder().name("Holds").links("card", "other").properties(
    (p) => p.name("what").of(Free), text("note")).unique("note").create();
  const Holder = new S.OfObject.Builder().name("Holder").ref().relations((r) => r.name("holds").of(Holds).me("card"),
    (r) => r.name("held").of(Holds).me("other")).create();
  for (const schema of [Holds, Holder]) store.register(schema);
  const target = B.Holder().create();
  const holder = B.Holder().holds((a: any) => a.insert((e: any) => e.other(target).what(phone).note("a")).insert(
    (e: any) => e.other(target).what(B.Phone().number("2").create()).note("b")).insert(
    (e: any) => e.other(target).what(1n).note("c")).insert((e: any) => e.other(target).note("d"))).create();
  assert(same(validate(Holder, holder), [
    "unique(note) violated: entries agreeing on ['card', 'other', 'what'] differ on ['note']"])); // equal values: one key
  for (const schema of [Holds, Holder]) other.register(schema);
  const snapshot = Plain.ToPlain(store).Reachable(Holder, holder);
  assert(equal((root_of(snapshot) as any).get("holds").map((e: any) => e.get("what") ?? null),
    [map({ Phone: { number: "2" } }), map({ Phone: { number: "2" } }), map({ int: 1n }), null]));
  assert(JSON.dumps(Plain.ToPlain(other).Reachable(Holder, Plain.FromPlain(other).Reachable(Holder, snapshot) as any))
    === JSON.dumps(snapshot));
  assert(Plain.FromPlain(store).Reachable(Holder, snapshot) === holder && holder.holds.length === 4); // reloading adds none
  assert(new C.OfObject(Card, card).compare(new C.OfObject(Card, Plain.FromPlain(other).OfObject(Card,
    Plain.ToPlain(store).OfObject(Card, card)) as any)) === 0); // a container of values of any type, element by element
}

## ANY-05 · Modules write a value of any type's types, by name or inline, and read them back

In [ ]:
{
  const module = Modules.module(store, [Phone, Card]);
  const definition = (root_of(Plain.ToPlain(store)(S.Module.Schema, module as any)) as any).get("schemas").get("Card");
  assert(equal(definition.get("object").get("properties").slice(0, 2), map([
    { name: "one", type: { any: { types: [{ scalar: { format: "basic", token: "int" } },
      { scalar: { format: "basic", token: "str" } }, { named: { name: "Phone" } }] } } },
    { name: "free", type: { any: {} } }])));
  const back = Modules.schemas(store, module) as Map<string, any>;
  const [one, free] = ["one", "free"].map((n) => back.get("Card").properties.get(n).type);
  assert(one instanceof S.OfAny.Value && one.types[2] === back.get("Phone") && one.types[0]!.equals(Int) && free.types.length === 0);
  assert(equal(Modules.reference(new S.OfAny.Value([Int], { description: "Counted" })), map({ any: { types: [
    { scalar: { format: "basic", token: "int" } }], description: "Counted" } })));
}